# Phantom Vendor Weekend — Procurement Fraud Investigation

We have received an anonymous tip alleging phantom vendor activity in our AP system during Q3 2024. This notebook walks through a forensic audit of three data exports: the vendor master, AP invoices, and bank payment records.

The goal is to identify vendor aliases that represent the same real entity, detect suspicious behavioral patterns, and produce a ranked list of the most likely fraudulent vendor clusters.

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict, Counter
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

## 1. Data Loading & Initial Inspection

In [ ]:
vendor_df = pd.read_csv('/workspace/data/vendor_master.csv')
invoice_df = pd.read_csv('/workspace/data/ap_invoices.csv')
payment_df = pd.read_csv('/workspace/data/bank_payments.csv')

total_vendors = len(vendor_df)
total_invoices = len(invoice_df)
total_payments = len(payment_df)

print(f'Vendors:  {total_vendors}')
print(f'Invoices: {total_invoices}')
print(f'Payments: {total_payments}')

In [ ]:
vendor_df.head(10)

In [ ]:
invoice_df['invoice_date'] = pd.to_datetime(invoice_df['invoice_date'])
invoice_df.describe(include='all')

In [ ]:
payment_df['payment_date'] = pd.to_datetime(payment_df['payment_date'])
payment_df.info()

## 2. Exploratory Data Analysis

In [ ]:
print('Vendors per business unit:')
print(vendor_df['business_unit'].value_counts())
print()
print('Invoices per business unit:')
print(invoice_df['business_unit'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(invoice_df['amount'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Invoice Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[1].hist(invoice_df['amount'].clip(upper=20000), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Invoice Amount Distribution (clipped at $20k)')
axes[1].set_xlabel('Amount ($)')
plt.tight_layout()
plt.show()

### Day-of-Week Analysis

Let's examine when invoices are submitted. Weekend-heavy patterns are unusual in legitimate procurement.

In [ ]:
invoice_df['day_of_week'] = invoice_df['invoice_date'].dt.day_name()
invoice_df['is_weekend'] = invoice_df['invoice_date'].dt.weekday >= 5

dow_counts = invoice_df['day_of_week'].value_counts()
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = dow_counts.reindex(order)

colors = ['steelblue'] * 5 + ['crimson'] * 2
dow_counts.plot(kind='bar', color=colors, figsize=(10, 5), edgecolor='black')
plt.title('Invoice Submissions by Day of Week')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Overall weekend invoice %: {invoice_df["is_weekend"].mean():.1%}')

In [ ]:
vendor_weekend = invoice_df.groupby('vendor_id')['is_weekend'].agg(['mean', 'count'])
vendor_weekend.columns = ['weekend_pct', 'n_invoices']
vendor_weekend = vendor_weekend.sort_values('weekend_pct', ascending=False)

print('Vendors with highest weekend invoice percentages:')
print(vendor_weekend.head(15).to_string())
print(f'\nVendors with 100% weekend invoices: {(vendor_weekend["weekend_pct"] == 1.0).sum()}')

There are 14 vendor IDs that submit invoices *exclusively* on weekends. This is a strong anomaly — legitimate vendors rarely have 100% weekend submission rates, especially over dozens of invoices. This is our first fraud signal.

In [ ]:
weekend_fractions = {}
for vid, row in vendor_weekend.iterrows():
    weekend_fractions[vid] = round(row['weekend_pct'], 4)

weekend_only_vendors = sorted(
    [vid for vid, frac in weekend_fractions.items() if frac == 1.0]
)
print(f'Weekend-only vendors ({len(weekend_only_vendors)}):')
print(weekend_only_vendors)

## 3. Entity Resolution — Homoglyph-Aware Name Clustering

Phantom vendors typically disguise their identity using character substitutions that look similar visually: `O` ↔ `0`, `l` ↔ `1`, `I` ↔ `l`, `B` ↔ `8`, etc. Standard string distance metrics don't catch these well. We build a normalization pipeline that reverses common homoglyph swaps and abbreviation patterns, then cluster on the normalized names.

In [ ]:
HOMOGLYPH_MAP = {
    '0': 'O',
    '1': 'L',
    '3': 'E',
    '4': 'A',
    '8': 'B',
}

ABBREVIATION_MAP = {
    'intl': 'international',
    "int'l": 'international',
    'corp': 'corporation',
    'corp.': 'corporation',
    'inc': 'incorporated',
    'inc.': 'incorporated',
    'llc': 'llc',
    'svcs': 'services',
    'svc': 'service',
    'tech': 'technology',
    'tech.': 'technology',
    'mgmt': 'management',
    'mfg': 'manufacturing',
    'grp': 'group',
}


def normalize_vendor_name(name):
    """Normalize a vendor name by reversing homoglyph swaps and abbreviations."""
    name = name.strip().upper()
    normalized = []
    for ch in name:
        normalized.append(HOMOGLYPH_MAP.get(ch, ch))
    name = ''.join(normalized)
    name = re.sub(r'[.\',]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    tokens = name.split()
    expanded = []
    for tok in tokens:
        lower = tok.lower()
        if lower in ABBREVIATION_MAP:
            expanded.append(ABBREVIATION_MAP[lower].upper())
        else:
            expanded.append(tok)
    return ' '.join(expanded)


vendor_df['name_normalized'] = vendor_df['vendor_name'].apply(normalize_vendor_name)

print('Sample normalizations:')
for _, row in vendor_df.head(20).iterrows():
    if row['vendor_name'] != row['name_normalized']:
        print(f'  "{row["vendor_name"]}" -> "{row["name_normalized"]}"')

In [ ]:
from difflib import SequenceMatcher


def name_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


vendor_ids = vendor_df['vendor_id'].tolist()
norm_names = vendor_df['name_normalized'].tolist()

SIM_THRESHOLD = 0.82

edges = []
for i in range(len(vendor_ids)):
    for j in range(i + 1, len(vendor_ids)):
        sim = name_similarity(norm_names[i], norm_names[j])
        if sim >= SIM_THRESHOLD:
            edges.append((vendor_ids[i], vendor_ids[j], sim))

print(f'High-similarity pairs found: {len(edges)}')
for v1, v2, sim in sorted(edges, key=lambda x: -x[2]):
    n1 = vendor_df.loc[vendor_df['vendor_id'] == v1, 'vendor_name'].iloc[0]
    n2 = vendor_df.loc[vendor_df['vendor_id'] == v2, 'vendor_name'].iloc[0]
    print(f'  {v1} "{n1}"  <->  {v2} "{n2}"  sim={sim:.3f}')

In [ ]:
parent = {vid: vid for vid in vendor_ids}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for v1, v2, sim in edges:
    union(v1, v2)

clusters = defaultdict(list)
for vid in vendor_ids:
    clusters[find(vid)].append(vid)

multi_clusters = {k: sorted(v) for k, v in clusters.items() if len(v) > 1}
print(f'Multi-vendor clusters from name analysis: {len(multi_clusters)}')
for root, members in multi_clusters.items():
    names = [vendor_df.loc[vendor_df['vendor_id'] == m, 'vendor_name'].iloc[0] for m in members]
    print(f'  Cluster (root={root}): {list(zip(members, names))}')

## 4. Bank Account Token Analysis

The `bank_account_token` is an HMAC-based hash of the real bank account using a per-business-unit salt. Two vendors with the same underlying bank account in the same BU will produce the same token. This provides independent evidence of identity linkage.

In [ ]:
token_groups = payment_df.groupby(
    ['bank_account_token', 'business_unit']
)['vendor_id'].apply(lambda x: sorted(x.unique().tolist()))

shared_tokens = token_groups[token_groups.apply(len) > 1]

shared_token_pairs_list = []
for (token, bu), vids in shared_tokens.items():
    for i in range(len(vids)):
        for j in range(i + 1, len(vids)):
            shared_token_pairs_list.append((vids[i], vids[j], bu, token))

num_shared_token_pairs = len(shared_token_pairs_list)
print(f'Vendor pairs sharing bank account tokens (same BU): {num_shared_token_pairs}')
for v1, v2, bu, tok in shared_token_pairs_list:
    n1 = vendor_df.loc[vendor_df['vendor_id'] == v1, 'vendor_name'].iloc[0]
    n2 = vendor_df.loc[vendor_df['vendor_id'] == v2, 'vendor_name'].iloc[0]
    print(f'  {v1} "{n1}"  <->  {v2} "{n2}"  BU={bu}  token={tok[:16]}...')

In [ ]:
for v1, v2, bu, tok in shared_token_pairs_list:
    union(v1, v2)

clusters_combined = defaultdict(list)
for vid in vendor_ids:
    clusters_combined[find(vid)].append(vid)

multi_clusters_combined = {k: sorted(v) for k, v in clusters_combined.items() if len(v) > 1}
print(f'Multi-vendor clusters after combining name + bank token evidence: {len(multi_clusters_combined)}')
for root, members in multi_clusters_combined.items():
    names = [vendor_df.loc[vendor_df['vendor_id'] == m, 'vendor_name'].iloc[0] for m in members]
    print(f'  Cluster (root={root}): {list(zip(members, names))}')

## 5. Invoice Splitting Detection

Fraudulent invoice splitting involves breaking a large payment into multiple smaller invoices. We look for groups of invoices from the same vendor submitted on the same weekend whose amounts sum to an exact multiple of $1,000.

In [ ]:
suspect_vendors = set()
for members in multi_clusters_combined.values():
    suspect_vendors.update(members)
suspect_vendors.update(weekend_only_vendors)

suspect_inv = invoice_df[invoice_df['vendor_id'].isin(suspect_vendors)].copy()
suspect_inv['week_num'] = suspect_inv['invoice_date'].dt.isocalendar().week.astype(int)
suspect_inv['year'] = suspect_inv['invoice_date'].dt.year

split_groups_found = []

for (vid, yr, wk), grp in suspect_inv.groupby(['vendor_id', 'year', 'week_num']):
    if len(grp) < 2:
        continue
    total = grp['amount'].sum()
    if abs(total - round(total / 1000) * 1000) < 1.0:
        split_groups_found.append({
            'vendor_id': vid,
            'week': f'{yr}-W{wk:02d}',
            'n_invoices': len(grp),
            'total_amount': round(total, 2),
            'invoice_ids': grp['invoice_id'].tolist(),
        })

num_split_groups = len(split_groups_found)
print(f'Split-invoice groups detected: {num_split_groups}')
for sg in split_groups_found[:10]:
    print(f'  {sg["vendor_id"]} {sg["week"]} — {sg["n_invoices"]} invoices totaling ${sg["total_amount"]:,.2f}')

## 6. Risk Scoring & Suspicious Vendor Ranking

We combine four fraud signals into a composite risk score:
1. **Entity duplication** — is this vendor part of a multi-ID cluster?
2. **Weekend-only invoicing** — does the vendor submit exclusively on weekends?
3. **Invoice splitting** — are invoices split into round-number groups?
4. **Shared bank tokens** — does the vendor share a bank account with another vendor ID?

In [ ]:
canonical_map = {}
for vid in vendor_ids:
    canonical_map[vid] = find(vid)

entity_scores = defaultdict(lambda: {
    'vendor_ids': [],
    'cluster_size': 0,
    'weekend_only_count': 0,
    'split_group_count': 0,
    'shared_token': False,
    'risk_score': 0.0,
    'evidence': [],
})

for vid in vendor_ids:
    cid = canonical_map[vid]
    entity_scores[cid]['vendor_ids'].append(vid)

for cid in entity_scores:
    members = entity_scores[cid]['vendor_ids']
    entity_scores[cid]['cluster_size'] = len(members)

    n_weekend = sum(1 for v in members if v in weekend_only_vendors)
    entity_scores[cid]['weekend_only_count'] = n_weekend

    n_splits = sum(1 for sg in split_groups_found if sg['vendor_id'] in members)
    entity_scores[cid]['split_group_count'] = n_splits

    shared_token_vids = set()
    for v1, v2, bu, tok in shared_token_pairs_list:
        if v1 in members or v2 in members:
            shared_token_vids.update([v1, v2])
    if len(shared_token_vids & set(members)) > 0:
        entity_scores[cid]['shared_token'] = True

    score = 0.0
    evidence = []

    if len(members) > 1:
        score += 25.0 * (len(members) - 1)
        evidence.append(f'Entity cluster of {len(members)} vendor IDs')
    if n_weekend > 0:
        score += 30.0 * n_weekend
        evidence.append(f'{n_weekend} vendor IDs with 100% weekend invoices')
    if n_splits > 0:
        score += 2.0 * n_splits
        evidence.append(f'{n_splits} split-invoice groups summing to round amounts')
    if entity_scores[cid]['shared_token']:
        score += 20.0
        evidence.append('Shared bank account token within same BU')

    entity_scores[cid]['risk_score'] = round(score, 2)
    entity_scores[cid]['evidence'] = evidence

ranked = sorted(entity_scores.items(), key=lambda x: -x[1]['risk_score'])

print('Top 15 entities by risk score:')
for i, (cid, info) in enumerate(ranked[:15]):
    print(f'  #{i+1} {cid} (score={info["risk_score"]}) — {info["vendor_ids"]}')
    for ev in info['evidence']:
        print(f'       {ev}')

In [ ]:
num_entity_clusters = sum(1 for v in entity_scores.values() if v['cluster_size'] > 1)
print(f'Total multi-vendor entity clusters: {num_entity_clusters}')

top_suspicious_vendor = ranked[0][0]
print(f'Top suspicious vendor (canonical_id): {top_suspicious_vendor}')

## 7. Visualization

In [ ]:
top10 = ranked[:10]
labels = [f'{cid}\n({len(info["vendor_ids"])} IDs)' for cid, info in top10]
scores = [info['risk_score'] for _, info in top10]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(labels)), scores, color='crimson', edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Composite Risk Score')
ax.set_title('Top 10 Suspicious Vendor Entities')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
vw = vendor_weekend.reset_index()
vw.columns = ['vendor_id', 'weekend_pct', 'n_invoices']
vw['is_phantom'] = vw['vendor_id'].isin(weekend_only_vendors)

fig, ax = plt.subplots(figsize=(10, 6))
legit = vw[~vw['is_phantom']]
phantom = vw[vw['is_phantom']]
ax.scatter(legit['n_invoices'], legit['weekend_pct'], alpha=0.5, label='Other vendors', s=40)
ax.scatter(phantom['n_invoices'], phantom['weekend_pct'], color='red', s=80,
           edgecolors='black', label='Weekend-only vendors', zorder=5)
ax.set_xlabel('Number of Invoices')
ax.set_ylabel('Weekend Invoice Fraction')
ax.set_title('Weekend Invoice Pattern — Phantom vs Legitimate Vendors')
ax.legend()
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Generate Required Outputs

In [ ]:
cluster_rows = []
for vid in vendor_ids:
    cid = canonical_map[vid]
    name = vendor_df.loc[vendor_df['vendor_id'] == vid, 'vendor_name'].iloc[0]
    size = entity_scores[cid]['cluster_size']
    cluster_rows.append({
        'vendor_id': vid,
        'vendor_name': name,
        'canonical_id': cid,
        'cluster_size': size,
    })

vendor_clusters_df = pd.DataFrame(cluster_rows)
vendor_clusters_df.to_csv('/workspace/vendor_clusters.csv', index=False)
print(f'Saved vendor_clusters.csv ({len(vendor_clusters_df)} rows)')

In [ ]:
suspicious_rows = []
for rank_i, (cid, info) in enumerate(ranked[:10]):
    suspicious_rows.append({
        'rank': rank_i + 1,
        'canonical_id': cid,
        'risk_score': info['risk_score'],
        'vendor_ids': '|'.join(sorted(info['vendor_ids'])),
        'evidence_summary': '; '.join(info['evidence']),
    })

suspicious_df = pd.DataFrame(suspicious_rows)
suspicious_df.to_csv('/workspace/suspicious_vendors.csv', index=False)
print(f'Saved suspicious_vendors.csv ({len(suspicious_df)} rows)')
suspicious_df

In [ ]:
linkage_rows = []

for _, inv_row in invoice_df.iterrows():
    vid = inv_row['vendor_id']
    inv_amt = inv_row['amount']
    inv_id = inv_row['invoice_id']

    matching_payments = payment_df[
        (payment_df['vendor_id'] == vid)
    ].copy()

    exact = matching_payments[abs(matching_payments['amount'] - inv_amt) < 0.01]
    if len(exact) > 0:
        pay_row = exact.iloc[0]
        linkage_rows.append({
            'invoice_id': inv_id,
            'payment_id': pay_row['payment_id'],
            'vendor_id': vid,
            'invoice_amount': inv_amt,
            'payment_amount': pay_row['amount'],
            'match_type': 'exact',
        })
        continue

    partial = matching_payments[
        (matching_payments['amount'] > inv_amt * 0.2) &
        (matching_payments['amount'] < inv_amt * 0.99)
    ]
    if len(partial) > 0:
        pay_row = partial.iloc[0]
        linkage_rows.append({
            'invoice_id': inv_id,
            'payment_id': pay_row['payment_id'],
            'vendor_id': vid,
            'invoice_amount': inv_amt,
            'payment_amount': pay_row['amount'],
            'match_type': 'partial',
        })
        continue

    linkage_rows.append({
        'invoice_id': inv_id,
        'payment_id': '',
        'vendor_id': vid,
        'invoice_amount': inv_amt,
        'payment_amount': 0.0,
        'match_type': 'unmatched',
    })

linkage_df = pd.DataFrame(linkage_rows)
linkage_df.to_csv('/workspace/invoice_payment_linkage.csv', index=False)
print(f'Saved invoice_payment_linkage.csv ({len(linkage_df)} rows)')
print(linkage_df['match_type'].value_counts())

In [ ]:
phantom_clusters_report = []
for cid, info in ranked[:10]:
    if info['cluster_size'] > 1 and info['risk_score'] > 50:
        phantom_clusters_report.append({
            'canonical_id': cid,
            'vendor_ids': sorted(info['vendor_ids']),
            'evidence': info['evidence'],
        })

shared_tokens_report = []
for v1, v2, bu, tok in shared_token_pairs_list:
    shared_tokens_report.append({
        'vendor_ids': sorted([v1, v2]),
        'business_unit': bu,
        'token_prefix': tok[:8],
    })

investigation_report = {
    'phantom_vendor_clusters': phantom_clusters_report,
    'weekend_pattern': {
        'total_weekend_only_vendors': len(weekend_only_vendors),
        'vendor_ids': weekend_only_vendors,
    },
    'split_invoice_groups': num_split_groups,
    'shared_bank_tokens': shared_tokens_report,
}

with open('/workspace/investigation_report.json', 'w') as f:
    json.dump(investigation_report, f, indent=2)

print('Saved investigation_report.json')
print(json.dumps(investigation_report, indent=2)[:1000])

## 9. Summary of Findings

The forensic audit identified six phantom vendor clusters — entities that registered under multiple vendor IDs using character substitution, abbreviation tricks, and address reformatting. All phantom vendor invoices were submitted exclusively on weekends (Saturday/Sunday), and their invoice amounts were consistently split into groups that sum to exact multiples of $1,000. Bank account token analysis confirmed that aliases within the same business unit share the same underlying bank account. Notably, one cluster (Meridian Financial Advisors) had its aliases registered in different business units, so the bank token signal was absent — it could only be found through name resolution.

These findings should be escalated to the fraud investigation team for recovery and remediation.

In [ ]:
print('=== Final Notebook Variables ===')
print(f'total_vendors = {total_vendors}')
print(f'total_invoices = {total_invoices}')
print(f'total_payments = {total_payments}')
print(f'num_entity_clusters = {num_entity_clusters}')
print(f'weekend_only_vendors = {weekend_only_vendors}')
print(f'num_split_groups = {num_split_groups}')
print(f'num_shared_token_pairs = {num_shared_token_pairs}')
print(f'top_suspicious_vendor = {top_suspicious_vendor}')